# 🍁 LeafID: Colab runner

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run sections **1–3** (about 5 min).
After that, run only the cells you own.

**Got disconnected?** Run sections 1–3 again, then re-run the same training cell. Every run uses `--resume`,
so it continues from the last finished epoch saved on your Google Drive.

| Section | What | When |
|---|---|---|
| 1. Setup | clone/update repo, install | every session |
| 2. Drive | where checkpoints + reports are saved | every session |
| 3. Data | download LeafSnap, build the fast image cache | every session |
| 4. Smoke test | 2-epoch run that checks the whole pipeline | once, or after code changes |
| 5. Experiments | the real training runs | one cell per run, whoever owns it |
| 6. Compare | evaluate on test + results table + curves (yours + teammates') | after runs finish |
| 6b. Grad-CAM | figure of a run's most-confused species pair | error analysis / slides |
| 7. App | launch the Gradio demo | demo / screenshots / video |

## 1. Setup

In [ ]:
import os, sys

BRANCH = "main"  # @param {type:"string"}
REPO_URL = "https://github.com/akshayanbalathas/leafid.git"

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"
if not os.path.exists("/content/leafid"):
    !git clone -q {REPO_URL} /content/leafid
%cd /content/leafid
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}
!pip install -q -r requirements.txt && pip install -q -e .
sys.path.insert(0, "/content/leafid/src")  # so this notebook can import leafid without a restart
!git log --oneline -1

## 2. Google Drive
Checkpoints and reports are written straight to Drive **during** training, so a disconnect loses at most one epoch.
Untick `USE_DRIVE` only for throwaway experiments.

In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT = "/content/drive/MyDrive/leafid"
else:
    OUTPUT = "/content/leafid_output"  # deleted when the session ends
os.makedirs(OUTPUT, exist_ok=True)
os.environ["LEAFID_OUTPUT"] = OUTPUT  # picked up by every !python -m leafid... command below
print("checkpoints + reports ->", OUTPUT)

## 3. Data
Downloads LeafSnap (~880 MB, public, no Kaggle login needed), then shrinks every image once to 256 px into
`/content/cache_256`. Training then skips decoding 800×600 JPEGs every epoch, which is the bottleneck on Colab's 2 CPUs.
The cache lives on the local disk (fast) and is rebuilt each session in a few minutes. **Don't** put it on Drive.

In [ ]:
import kagglehub

try:  # optional: only needed if Kaggle ever refuses anonymous downloads (add in the 🔑 Secrets panel)
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    pass

os.environ["LEAFID_DATA"] = kagglehub.dataset_download("xhlulu/leafsnap-dataset")
os.environ["LEAFID_CACHE"] = "/content/cache_256"
assert os.path.exists("splits/splits.csv"), "splits/splits.csv missing: run section 1 again to pull main"
print("dataset:", os.environ["LEAFID_DATA"])

!python -m leafid.cache

## 4. Smoke test (~10 min)
Checks data loading, both training stages, checkpointing and evaluation before anyone spends hours training.
Saves to `/content/smoke`, not Drive. Note the **seconds per epoch**: it tells you how many epochs fit in a session.
Field val top-1 just needs to be clearly above chance (1/185 ≈ 0.5%).

In [ ]:
from leafid.data import image_root, make_loader

x, y = next(iter(make_loader("train", train=True, batch_size=8)))
print("images from:", image_root())
print("batch:", tuple(x.shape), x.dtype, "labels:", y.tolist())  # expect (8, 3, 224, 224) torch.float32

!python -m leafid.train --model resnet50 --epochs 2 --freeze-epochs 1 --run smoke --save-dir /content/smoke
!python -m leafid.evaluate --checkpoint /content/smoke/checkpoints/smoke.pt --save-dir /content/smoke

## 5. Experiments
One cell per run. **Run only the ones you own**, so we don't duplicate GPU time across accounts.
Every cell ends in `--resume`: after a disconnect, run sections 1–3 and then the same cell again.

| Run | What it tests | Owner |
|---|---|---|
| `clip_zeroshot` | CLIP with **no training**: species names as text prompts (~5 min) | Akshayan |
| `simple_cnn` | small CNN from scratch (baseline) | Akshayan |
| `enhanced_cnn` | deeper CNN (2 convs per block, 512 features), from scratch | Allan |
| `res_cnn` | + residual connections, still from scratch | Allan |
| `resnet50_probe` | pretrained ImageNet features, only the head trains | TODO |
| `resnet50_full` | two-stage fine-tune (**the model the app uses**) | Akshayan |
| `resnet50_labonly` | trained on lab photos only → lab-to-field domain shift | TODO |

In [ ]:
# No training: CLIP matches each test photo against text prompts built from the 185 species names.
MODEL_ID = "openai/clip-vit-base-patch32"  # @param ["openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
RUN_NAME = "clip_zeroshot" if MODEL_ID.endswith("base-patch32") else "clip_zeroshot_L14"
!python -m leafid.zeroshot --model-id {MODEL_ID} --run {RUN_NAME}

In [ ]:
!python -m leafid.train --model simple_cnn --epochs 30 --lr 1e-3 --run simple_cnn --resume

In [ ]:
!python -m leafid.train --model enhanced_cnn --epochs 30 --lr 1e-3 --run enhanced_cnn --resume

In [ ]:
!python -m leafid.train --model res_cnn --epochs 30 --lr 1e-3 --run res_cnn --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 10 --freeze-epochs 10 --run resnet50_probe --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3 --run resnet50_full --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3 --train-sources lab --run resnet50_labonly --resume

## 6. Evaluate + compare
Evaluates every **finished** run on your Drive that hasn't been evaluated on the current split yet, then shows
the results table and validation curves. The table also includes teammates' runs that were committed to the
repo's `reports/` folder (column `from`), so after everyone's PRs are merged it shows all the experiments.

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt
from leafid.data import split_version
from leafid.models import MODEL_NAMES, count_parameters

OUT = os.environ["LEAFID_OUTPUT"]
SPLIT = split_version()

def up_to_date(run, ckpt):
    """metrics.json exists, is newer than the checkpoint and was made on the current split."""
    m = f"{OUT}/reports/{run}/metrics.json"
    return (os.path.exists(m) and os.path.getmtime(m) > os.path.getmtime(ckpt)
            and json.load(open(m)).get("split_version") == SPLIT)

for ckpt in sorted(glob.glob(f"{OUT}/checkpoints/*.pt")):
    run = os.path.basename(ckpt)[:-3]
    if run.endswith("_last") or os.path.exists(f"{OUT}/checkpoints/{run}_last.pt"):
        print(f"skipping {run}: still in progress")
    elif up_to_date(run, ckpt):
        print(f"skipping {run}: already evaluated")
    else:
        !python -m leafid.evaluate --checkpoint "{ckpt}"

# Teammates' results committed to the repo, then yours on Drive (yours win for the same run name).
found = {}
for where, base in (("repo", "/content/leafid/reports"), ("drive", f"{OUT}/reports")):
    for p in glob.glob(f"{base}/*/metrics.json"):
        m = json.load(open(p))
        if m.get("split_version") == SPLIT:  # results from older splits aren't comparable
            found[p.split("/")[-2]] = (where, os.path.dirname(p), m)

def params(model):  # size of the architecture, for a fair comparison (CLIP rows: not a trained classifier)
    return round(count_parameters(model) / 1e6, 1) if model in MODEL_NAMES else None

display(pd.DataFrame([{"run": run, "from": where, "model": m["model"], "params_M": params(m["model"]),
                       "lab_top1": m["lab"]["top1"],
                       "field_top1": m["field"]["top1"], "field_top5": m["field"]["top5"]}
                      for run, (where, _, m) in sorted(found.items())]).round(3))

plt.figure(figsize=(7, 4))
for run, (_, folder, _) in sorted(found.items()):
    if os.path.exists(f"{folder}/history.csv"):
        h = pd.read_csv(f"{folder}/history.csv")
        plt.plot(h["epoch"], h["val_field_top1"], marker="o", label=run)
plt.xlabel("epoch"); plt.ylabel("val field top-1"); plt.title("Validation accuracy on field photos")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 6b. Grad-CAM of the most-confused pair (~1 min, no training)
Run section 6 first. Shows the field test photos of the species a model confuses most (mistakes first), where it
looked (Grad-CAM), and training photos of both species, and saves the figure to `reports/<run>/` for the README
and slides. This is how we found that LeafSnap files identical photos under **two different species**, which
split v2 removes (see `splits/dropped.csv`). Leave `PAIR` empty for the most-confused pair, or name one from
the run's `confusions.csv`, e.g. `catalpa_speciosa catalpa_bignonioides`.

In [ ]:
from IPython.display import Image, display

RUN = "resnet50_full"  # @param {type:"string"}
PAIR = ""  # @param {type:"string"}

pair = PAIR.split()
png = f"{os.environ['LEAFID_OUTPUT']}/reports/{RUN}/" + (f"gradcam_{pair[0]}__{pair[1]}.png" if pair
                                                         else "gradcam_top_confusion.png")
if os.path.exists(png):
    os.remove(png)  # so a failed run can't show an old figure
FLAGS = "--pair " + " ".join(pair) if pair else ""
!python -m leafid.explain --run {RUN} {FLAGS}
if os.path.exists(png):
    display(Image(png))

## 7. Launch the app
Opens a public `gradio.live` link. The first launch downloads CLIP and the LLM (a few minutes);
untick `USE_LLM` for a fast start. Stop the cell (■) when done: a running app keeps using your GPU quota.

In [ ]:
RUN = "resnet50_full"  # @param {type:"string"}
USE_LLM = True  # @param {type:"boolean"}

CKPT = f"{os.environ['LEAFID_OUTPUT']}/checkpoints/{RUN}.pt"
assert os.path.exists(CKPT), f"{CKPT} not found: train it first (section 5) or pick another RUN"
FLAGS = "" if USE_LLM else "--no-llm"
!python app/app.py --checkpoint "{CKPT}" --share {FLAGS}

## Getting results into the repo
Checkpoints stay on Drive (too big for GitHub). The **reports** are small: download
`MyDrive/leafid/reports/<run>/` (history.csv, metrics.json, confusion_field.png, confusions.csv, and any gradcam_*.png),
copy the folder into the repo's `reports/` folder locally (so it becomes `reports/<run>/`), and open a PR. The README results table is filled from `metrics.json`.